# TabICL — subsample-in-fold bagging for stacking (T4 x2, fe_v4_native)

Produces a **full 594k-row, leakage-free OOF vector** for TabICL so it can join the
stacking ensemble, upgrading the 50k-subsample EDA run `20260603-222236-617994`
(tabicl-eda). Instead of subsampling *before* the CV (which left 544k rows with no OOF
prediction), the subsampling moves **inside each fold**: fit on stratified 50k draws
from the fold-training rows only, predict the entire validation fold and test set.
Three bagged draws per fold are averaged (fits are cheap; prediction dominates the
runtime).

**Two upgrades vs the EDA parent.** (1) **Features:** `fe_v4_native` — the minimal
engineered set the GBDTs scored best with (`contract_x_internet`, `contract_x_payment`,
`AverageMonthly`) on **native category-dtype** categoricals instead of fe_v0 one-hot.
TabICL handles categorical columns directly (internal encoding by value), and ICL
transformers generally prefer raw categoricals over one-hot expansion. (2) **Both
T4s:** TabICL is single-device per model, so the wrapper fits an identical bag ensemble
on each GPU (fits are seconds) and splits the query rows across the two replicas at
predict time — ~2x on the prediction-bound wall time.

**Fold contract.** The outer splitter is the project-wide
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`. A check cell asserts the
on-platform `(id, fold)` assignment matches the committed
`experiments/cv_folds_seed42.csv.gz`, so the OOF is guaranteed stackable against the
local runs. **Push that file to GitHub master before running** — the clone must contain it.

**Notebook settings (right sidebar).** Accelerator → **GPU T4 x2** (not P100);
Internet → **On**; Add Input → **playground-series-s6e3**.

> ⏱️ Each fold predicts 3 bags × (~119k val + ~255k test) rows ≈ 1.1M queries with a
> 50k context, halved across two T4s — expect **roughly 2–3 h** total.
> **Smoke-test first**: set `subsample_n=2_000, n_bags=1` in `run_config['params']`,
> confirm one fold's timing and that *both* GPUs light up in `nvidia-smi`, then restore
> and Save & Run All.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell. (A plain
# `if REPO_ROOT not in sys.path` guard can leave a shadowing `src` ahead of ours.)
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...
Updating files: 100% (319/319), done.


CWD: /kaggle/working/Predict-Customer-Churn


In [3]:
!pip install -q tabicl        # Kaggle's GPU image already ships torch + CUDA

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())     # expect 2 on T4 x2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [4]:
# cuda.is_available() can return True on an incompatible GPU — run a real op on BOTH
# devices (predict_proba splits work across them, so both must be healthy).
import torch
for d in range(torch.cuda.device_count()):
    try:
        _ = (torch.randn(16, device=f"cuda:{d}") @ torch.randn(16, 16, device=f"cuda:{d}")).sum().item()
        print(f"cuda:{d} ({torch.cuda.get_device_name(d)}): compute OK")
    except Exception as e:
        print(f"cuda:{d} compute FAILED:", e)   # if this fails, switch to T4 x2 and restart

from tabicl import TabICLClassifier
print("TabICLClassifier imported OK")

cuda:0 (Tesla T4): compute OK
cuda:1 (Tesla T4): compute OK
TabICLClassifier imported OK


In [5]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild the
# NATIVE (category-dtype) frames from the attached competition CSVs — TabICL consumes
# category columns directly, so this is the same data path as the GBDT fe_v4 runs.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native', force=True)
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

Preprocessed and saved (native): train_df (594194, 21), test_df (254655, 20)
Loaded native: train_df (594194, 21), test_df (254655, 20)


### Feature engineering — fe_v4_native (minimal set)

Same `engineer_features` as the GBDT min3 runs (`lgbm-catreg-fe-min3`,
`catboost-gpu-fe-min3`): `AverageMonthly` plus the two low-cardinality crosses, all
row-wise (stateless) so there is no leakage when applied before the CV split. No
`tenure == 0` rows exist in the data, so `AverageMonthly` is always finite — relevant
here because TabICL rejects infinities (the GBDTs would have tolerated them).

In [6]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from src.tracking import DATA_DIR

DATA_VERSION = 'fe_v4_native'   # minimal engineered set on the native categorical base


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-wise (stateless) feature engineering — identical to the GBDT min3 runs."""
    df = df.copy()
    df['AverageMonthly'] = df['TotalCharges'] / df['tenure']
    df['contract_x_payment'] = (df['Contract'].astype(str) + ' | '
                                + df['PaymentMethod'].astype(str)).astype('category')
    df['contract_x_internet'] = (df['Contract'].astype(str) + ' | '
                                 + df['InternetService'].astype(str)).astype('category')
    return df


# Cache engineered parquets per DATA_VERSION (same convention as Experiments.ipynb).
fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')

Loaded cached FE: fe_v4_native


In [7]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score

from src.tracking import RUNS_DIR
from src.cv import run_cv_experiment, save_experiment

encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]

# TabICL auto-detects category-dtype columns and encodes them BY VALUE at fit time, so
# differing train/test category sets cannot cause a code mismatch (unlike XGBoost's
# raw-codes path, which needs explicit alignment in Experiments.ipynb).
cat_features = [c for c in encoded_features if str(X_train[c].dtype) == 'category']
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  '
      f'features: {len(encoded_features)}  categorical: {len(cat_features)}')

X_train: (594194, 22)  X_test: (254655, 22)  features: 22  categorical: 17


### Fold-contract check

Stacking aligns OOF vectors by row position across runs from different environments, so
this cell asserts the contract instead of assuming it: the on-platform regenerated rows
(by `id`) and the `StratifiedKFold(5, shuffle, seed 42)` fold assignment must exactly
match `experiments/cv_folds_seed42.csv.gz`, committed from the local environment by
`scripts/make_cv_folds.py`. If either assert fires, **stop** — do not save the run.

In [8]:
folds_ref = pd.read_csv('experiments/cv_folds_seed42.csv.gz')   # CWD = repo root

cv_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds_here = np.full(len(train_df), -1)
# X is only consulted for its length; zeros keep this independent of features.
for fold, (_, va_idx) in enumerate(cv_check.split(np.zeros(len(train_df)), train_df['Churn'])):
    folds_here[va_idx] = fold

assert (folds_ref['id'].to_numpy() == train_df['id'].to_numpy()).all(), \
    'Row order differs from local — OOF would NOT be stackable. Stop.'
assert (folds_ref['fold'].to_numpy() == folds_here).all(), \
    'Fold assignment differs from local — OOF would NOT be stackable. Stop.'
print('Fold contract OK: rows and folds match the committed local assignment.')

Fold contract OK: rows and folds match the committed local assignment.


### Subsample-in-fold bagging wrapper (dual-GPU)

`run_cv_experiment` calls `model.fit(X_train.iloc[tr_idx], ...)` with the fold's ~475k
training rows — far beyond a T4-comfortable TabICL context. The wrapper moves the
subsample *inside* `fit`: it draws `n_bags` independent stratified `subsample_n`-row
samples **from the rows it is given only** (never the fold's validation rows, so the
OOF stays leakage-free) and fits one TabICL per draw.

**Both T4s.** TabICL is single-device per model, so `fit` replicates the identical bag
ensemble on every device in `devices` (same subsample and seed per bag → interchangeable
replicas; fits cost seconds). `predict_proba` then splits the query rows contiguously
across devices and runs one thread per device — torch releases the GIL during CUDA work,
so the halves run concurrently for ~2x on the prediction-bound wall time.

Each half is bag-averaged in `chunk`-row slices — without chunking, a 254k-row call
makes TabICL allocate one ~22 GB column-embedding buffer and OOM the T4 (same VRAM math
as the `ChunkedTabICL` class in the tabicl-eda run); at 20k rows per call the buffer is
~1.7 GB, entirely on GPU. Chunking and row-splitting are exact: each query row attends
only to the fitted context.

Subclassing `BaseEstimator` makes `get_params()` return the clean constructor args, so
the run's `params.json` and `params_hash` stay informative.

In [9]:
from concurrent.futures import ThreadPoolExecutor

from sklearn.base import BaseEstimator, ClassifierMixin


class BaggedSubsampleTabICL(BaseEstimator, ClassifierMixin):
    """TabICL with fit-time stratified subsampling, bagging, and dual-GPU prediction
    (see markdown above)."""

    def __init__(self, subsample_n=50_000, n_bags=3, random_state=42,
                 chunk=20_000, devices=('cuda:0', 'cuda:1')):
        self.subsample_n  = subsample_n
        self.n_bags       = n_bags
        self.random_state = random_state
        self.chunk        = chunk
        self.devices      = devices

    def fit(self, X, y):
        X = X.reset_index(drop=True)
        y = y.reset_index(drop=True)
        # Sequential fits: the first one downloads the checkpoint, so no thread race.
        self.models_ = {dev: [] for dev in self.devices}
        for bag in range(self.n_bags):
            seed = self.random_state + bag   # independent draw + model seed per bag
            X_sub, _, y_sub, _ = train_test_split(
                X, y, train_size=self.subsample_n, stratify=y, random_state=seed,
            )
            X_sub = X_sub.reset_index(drop=True)
            y_sub = y_sub.reset_index(drop=True)
            for dev in self.devices:   # identical replica per device
                model = TabICLClassifier(device=dev, random_state=seed)
                model.fit(X_sub, y_sub)
                self.models_[dev].append(model)
        self.classes_ = self.models_[self.devices[0]][0].classes_
        return self

    def _predict_slice(self, dev, X):
        # Bag-average on one device, in chunks (bounds the per-call GPU buffers).
        total = None
        for model in self.models_[dev]:
            parts = [model.predict_proba(X.iloc[i:i + self.chunk])
                     for i in range(0, len(X), self.chunk)]
            proba = np.vstack(parts)
            total = proba if total is None else total + proba
        return total / len(self.models_[dev])

    def predict_proba(self, X):
        bounds = np.linspace(0, len(X), len(self.devices) + 1).astype(int)
        with ThreadPoolExecutor(max_workers=len(self.devices)) as pool:
            futures = [pool.submit(self._predict_slice, dev, X.iloc[bounds[i]:bounds[i + 1]])
                       for i, dev in enumerate(self.devices)]
            return np.vstack([f.result() for f in futures])

### Run configuration

The harness receives the **full** `X_train` this time — fold-level subsampling happens
inside the wrapper — so the logged OOF covers all 594k rows. `metric=accuracy_score`
mirrors the other runs; the harness always logs OOF ROC-AUC (the project's primary
metric) separately. `save_models=False` because TabICL is torch-backed (fragile to
`joblib.dump`).

In [10]:
from importlib.metadata import version

DEVICES = tuple(f'cuda:{i}' for i in range(max(1, torch.cuda.device_count())))

run_config = {
    'model_factory': lambda params: BaggedSubsampleTabICL(**params),
    'params': {
        'subsample_n':  50_000,   # SMOKE-TEST: 2_000
        'n_bags':       3,        # SMOKE-TEST: 1
        'random_state': 42,
        'chunk':        20_000,
        'devices':      DEVICES,
    },
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'tabicl-subfold-bag3',
    'notes': (
        'TabICL subsample-in-fold bagging for stacking: full 594k-row leakage-free OOF '
        'under the canonical StratifiedKFold(5, shuffle, seed 42). Per fold, 3 bags each '
        'fit on an independent stratified 50k subsample drawn from the fold-training rows '
        'only; val/test predictions are bag-averages, row-split across both T4s via '
        'per-device bag replicas. FE upgraded from the parent run\u2019s fe_v0 one-hot to '
        'fe_v4_native (contract_x_internet + contract_x_payment + AverageMonthly, native '
        'category dtype consumed directly). Fold assignment asserted against committed '
        'experiments/cv_folds_seed42.csv.gz. '
        f'tabicl={version("tabicl")}, torch={version("torch")}. '
        'Data regenerated on-platform; GPU run not bit-reproducible. '
        'Notebook: kaggle/predict-customer-churn-tabicl-subfold.ipynb.'
    ),
    'parent_run_id': '20260603-222236-617994',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

In [11]:
# Step 1 — Run the experiment. SMOKE-TEST FIRST: set subsample_n=2_000, n_bags=1 in the
# run_config params above, confirm one fold's timing and that BOTH GPUs show load in
# nvidia-smi, then restore 50_000 / 3 and Save & Run All. Keep n_splits=5 — the fold
# contract requires it.
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260612-031158-5adf7a
Tag:    tabicl-subfold-bag3

INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



tabicl-classifier-v2-20260212.ckpt:   0%|          | 0.00/110M [00:00<?, ?B/s]

Fold 0: accuracy=0.8588  roc_auc=0.9140  (fit 15.6s)
Fold 1: accuracy=0.8598  roc_auc=0.9149  (fit 11.1s)
Fold 2: accuracy=0.8589  roc_auc=0.9143  (fit 11.0s)
Fold 3: accuracy=0.8601  roc_auc=0.9152  (fit 10.1s)
Fold 4: accuracy=0.8596  roc_auc=0.9127  (fit 10.7s)

OOF accuracy: 0.8594
OOF ROC-AUC:  0.9142
Folds:        0.8594 ± 0.0005

Run complete. Call save_experiment(result) to log this run permanently.


In [12]:
# Step 2 — Save the run (optional). Review the OOF ROC-AUC printed above first, and
# only save if the fold-contract cell passed.
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260612-031158-5adf7a


### Build a submission (optional)

`test_proba_mean` is the fold-bagged (5 folds × 3 bags) churn probability for the full
test set. The competition metric is ROC-AUC, so submit the probability directly.

In [13]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.050683
1  594195  0.000656
2  594196  0.075536
3  594197  0.001943
4  594198  0.433449
wrote /kaggle/working/submission.csv (254655, 2)


### Bundle run artifacts into one zip

Zips the run directory and `runs.csv` into a single archive on the Output tab. To fold
the run back into the local repo, follow **§7–8 of `docs/kaggle_gpu_workflow.md`**, then
rerun `scripts/check_oof_alignment.py` with this run added as the post-merge gate.

In [14]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) heavy run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) the master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)

wrote /kaggle/working/20260612-031158-5adf7a_bundle.zip
